## Fully Connected Neural Network (Baseline)

This model serves as a baseline for text classification.  
Texts are represented using fixed-size vector representations, and the model does not take word order into account.  
As a result, this approach cannot capture sequential or contextual dependencies between words, which limits its performance on emotion classification.


In [1]:
import pandas as pd
import numpy as np
import random
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
import joblib

SEED = 42
np.random.seed(SEED)
random.seed(SEED)


In [2]:
BASE_PATH = "../dataset"

train_df = pd.read_csv(os.path.join(BASE_PATH, "train_clean.csv"))
test_df = pd.read_csv(os.path.join(BASE_PATH, "test_clean.csv"))

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()


Train shape: (16000, 2)
Test shape: (2000, 2)


,clean_text,label
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [3]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["label"])
y_test = label_encoder.transform(test_df["label"])

print("Encoded labels:", label_encoder.classes_)


Encoded labels: ['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [4]:
tfidf_clf = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=20000,
                ngram_range=(1, 2),
                stop_words=None
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=SEED,
                n_jobs=-1
            ),
        ),
    ]
)


In [5]:
tfidf_clf.fit(train_df["clean_text"], y_train)
print("TF-IDF baseline model trained.")


/Users/chiranthanshivakumar/NLP/nlp_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


TF-IDF baseline model trained.


In [6]:
y_pred = tfidf_clf.predict(test_df["clean_text"])

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1-score (weighted): {f1:.4f}")


Test Accuracy: 0.8340
Test F1-score (weighted): 0.8237


In [7]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)


              precision    recall  f1-score   support

       anger       0.91      0.74      0.82       275
        fear       0.87      0.72      0.79       224
         joy       0.78      0.97      0.86       695
        love       0.84      0.51      0.64       159
     sadness       0.87      0.91      0.89       581
    surprise       0.95      0.27      0.42        66

    accuracy                           0.83      2000
   macro avg       0.87      0.69      0.74      2000
weighted avg       0.84      0.83      0.82      2000



### Model Limitation

This fully connected baseline model relies on TF-IDF features and therefore ignores word order and long-range dependencies between tokens. As a result, it cannot capture sequential or contextual patterns in the text, which motivates the use of recurrent and Transformer-based architectures in the following experiments.


### Baseline Model Observations

- The TF-IDF + FC model performs reasonably well on frequent emotions
- It ignores word order and long-term dependencies
- Errors often occur for emotions expressed through context or negation

This confirms the limitations of bag-of-words representations
for emotion classification.


In [8]:
MODEL_PATH = "../models"

os.makedirs(MODEL_PATH, exist_ok=True)

joblib.dump(tfidf_clf, os.path.join(MODEL_PATH, "fc_baseline.pkl"))
joblib.dump(
    tfidf_clf.named_steps["tfidf"],
    os.path.join(MODEL_PATH, "tfidf_vectorizer.pkl")
)
joblib.dump(
    label_encoder,
    os.path.join(MODEL_PATH, "label_encoder.pkl")
)

print("Baseline model and vectorizer saved.")


Baseline model and vectorizer saved.


In [9]:
assert os.path.exists(os.path.join(MODEL_PATH, "fc_baseline.pkl"))
assert os.path.exists(os.path.join(MODEL_PATH, "tfidf_vectorizer.pkl"))
assert os.path.exists(os.path.join(MODEL_PATH, "label_encoder.pkl"))

print("Notebook 2 verified successfully.")


Notebook 2 verified successfully.


### Conclusion

The TF-IDF + FC model provides a strong non-sequential baseline.
Its limitations motivate the use of LSTM, BiLSTM, and Transformer
models explored in the following notebooks.
